# Hello, Numerai

Hello and welcome to the Numerai Data Science Tournament!

This notebook is designed to help you build your first machine learning model and start competing in the tournament.

In this notebook we will
1. Download and explore the Numerai dataset
2. Train your first machine learning model
3. Evaluate it with `NCORR` and benchmark contribution
4. Deploy it to start making live submissions

In [ ]:
!python --version

In [ ]:
# Install dependencies
!pip install -q --upgrade numerapi pandas==2.3.1 pyarrow matplotlib lightgbm scipy cloudpickle==3.1.1

# Inline plots
%matplotlib inline

## 1. Dataset

The Numerai dataset is a tabular dataset that describes the stock market over time. Each row represents a stock on a particular date and includes a ticker.

This dataset uses the same stock universe, target, and neutralizers as Signals.

### Listing the datasets
Firstly, take a look at the files Numerai offers below:

In [ ]:
# Initialize NumerAPI - the official Python API client for Numerai
from numerapi import NumerAPI
napi = NumerAPI()

# List the datasets and available versions
all_datasets = napi.list_datasets()
dataset_versions = list(set(d.split('/')[0] for d in all_datasets))
print("Available versions:\n", dataset_versions)

# Set data version to one of the latest datasets
DATA_VERSION = "v6.0"

# Print all files available for download for our version
current_version_files = [f for f in all_datasets if f.startswith(DATA_VERSION)]
print("Available", DATA_VERSION, "files:\n", current_version_files)

### Downloading datasets

The `features.json` file contains metadata about the features and targets available for training.

Let's download it and take a look:

In [ ]:
import json

# Download the feature metadata file
napi.download_dataset(f"{DATA_VERSION}/features.json")

# Read the metadata and display
feature_metadata = json.load(open(f"{DATA_VERSION}/features.json"))
for metadata in feature_metadata:
  print(metadata, len(feature_metadata[metadata]))

### Feature Sets & Groups
As you can see there are many features and targets to choose from.

Instead of training a model on all 1,018 features, let's pick a subset of features to analyze.

Here are the starter sets Numerai offers:

- `small` contains a minimal subset of features

- `medium` contains a broader set of features

- `all` contains every feature

Let's compare their sizes:

In [ ]:
feature_sets = feature_metadata["feature_sets"]
for feature_set_name in ["small", "medium", "all"]:
  print(feature_set_name, len(feature_sets[feature_set_name]))

We will use the `small` feature set.

For this notebook, we will train on a sample of the training data. Using more rows takes longer and requires more memory.

In [ ]:
import pandas as pd

# Define our feature set. feature_country is text; this model uses numeric features.
feature_set = [feature for feature in feature_sets["small"] if feature != "feature_country"]
assert feature_set, "The small feature set has no numeric features"

# Download the training data - this will take a few minutes
napi.download_dataset(f"{DATA_VERSION}/train.parquet")

# Load the features and training target for every training date
train = pd.read_parquet(
    f"{DATA_VERSION}/train.parquet",
    columns=["numerai_ticker", "date", "target"] + feature_set,
).dropna(subset=["target"])
assert not train.empty, "The training file has no mature target rows"
last_train_date = pd.Timestamp(train["date"].max())
train = train.sample(n=min(200_000, len(train)), random_state=7)

### Training data

Each row represents a stock at a specific point in time:
- `numerai_ticker` identifies the stock
- `date` is the date
- `target` is a measure of future returns for that stock
- `features` describe the attributes of the stock for that date

In [ ]:
train

### Dates
The historical `date` values are one week apart.

Think of the rows within a `date` as a single example - throughout this notebook we score everything "per date". The rows per date below trace the size of Numerai's stock universe over time.

In [ ]:
# Plot the number of rows per date
train.groupby("date").size().plot(
    title="Number of rows per date",
    figsize=(5, 3),
    xlabel="Date"
)

### Target
The `target` measures stock-specific returns over the next 60 business days. It is an alias for `target_jupiter_60` in this dataset.

Target values are binned into 5 unequal bins: `0`, `0.25`, `0.5`, `0.75`, `1.0`. This heavy regularization avoids overfitting because the underlying values are extremely noisy.

In [ ]:
# Plot density histogram of the target
train["target"].plot(
  kind="hist",
  title="Target",
  figsize=(5, 3),
  xlabel="Value",
  density=True,
  bins=50
)

### Features
The `features` are quantitative attributes of each stock: fundamentals, technical signals, market data, analyst ratings, and much more. Numerai includes them because we believe they help predict the `target`.

When available, numeric feature values are binned into 5 equal-frequency bins: `0`, `1`, `2`, `3`, `4`. `2` is the middle bin.

## 2. Modeling
Our task: predict the `target` using the `features`.

### Model training

You are free to use any tool or framework, but here we use LightGBM - a popular choice amongst tournament participants. While your model trains, watch this [video](https://www.youtube.com/watch?v=w8Y7hY05z7k) from our Chief Scientist MDO on why tree-based models work so well on tabular data.

In [ ]:
# https://lightgbm.readthedocs.io/en/latest/pythonapi/lightgbm.LGBMRegressor.html
import lightgbm as lgb

model = lgb.LGBMRegressor(
  n_estimators=300,
  learning_rate=0.05,
  max_depth=5,
  num_leaves=2**5-1,
  colsample_bytree=0.1,
  verbosity=-1,
)

# This will take a few minutes per model 🍵
model.fit(
  train[feature_set],
  train["target"]
)

### Validation predictions

Now let's make out-of-sample predictions on the validation dataset.

To stay within Colab's memory limits we free the training data, select dates across the validation period, and predict one date at a time.

In [ ]:
# The target looks 60 business days ahead, so exclude validation dates
# within that horizon of the last training date.
del train

napi.download_dataset(f"{DATA_VERSION}/validation.parquet")
validation_dates = pd.to_datetime(pd.read_parquet(
    f"{DATA_VERSION}/validation.parquet", columns=["date"]
)["date"].unique())
eligible_dates = sorted(date for date in validation_dates if date > last_train_date + pd.offsets.BDay(60))
assert eligible_dates, "No validation dates remain after the embargo"

# Sample dates across the full window to keep the backtest practical in Colab.
scoring_dates = [date.strftime("%Y-%m-%d") for date in eligible_dates[::16]]
validation = pd.read_parquet(
    f"{DATA_VERSION}/validation.parquet",
    columns=["numerai_ticker", "date", "data_type", "target"] + feature_set,
    filters=[("date", "in", scoring_dates), ("data_type", "==", "validation")],
).dropna(subset=["target"])
assert not validation.empty, "No selected validation rows have mature targets"

# Generate predictions one date at a time
predictions = []
for date, date_data in validation.groupby("date"):
    predictions.append(pd.Series(model.predict(date_data[feature_set]), index=date_data.index))
validation["prediction"] = pd.concat(predictions)

# Keep only the columns we need for scoring
validation = validation[["numerai_ticker", "date", "prediction", "target"]]
validation

### Performance evaluation

Numerai's scores are designed to "align incentives": a model with good scores should help the hedge fund make good returns.

- `NCORR` (`neutral_correlation`): correlation with the target after removing exposure to the neutralizer matrix.

- Benchmark contribution (`neutral_contribution`): how much signal your model adds beyond a public benchmark, after neutralization.

We'll calculate both over the same sampled validation dates.

In [ ]:
# Install Numerai's open-source scoring tools
!pip install -q --no-deps numerai-tools==0.7.2

from numerai_tools.scoring import neutral_correlation, neutral_contribution

### NCORR: neutral correlation with the target

`NCORR` asks whether your predictions align with the target after removing exposure to the supplied neutralizers.

Always score each `date` independently - "per date" metrics are what matter, not one score across the whole validation set.

And yes, the scores are low: +/- 0.05 is normal in quantitative finance. This is why we call Numerai the "hardest data science tournament on the planet".

In [ ]:
# Download the neutralizer matrix for the validation dates
napi.download_dataset(f"{DATA_VERSION}/validation_neutralizer.parquet")
neutralizers = pd.read_parquet(
    f"{DATA_VERSION}/validation_neutralizer.parquet",
    filters=[("date", "in", scoring_dates)],
)
neutralizer_columns = [column for column in neutralizers if column.startswith("neutralizer_column_")]
assert neutralizer_columns, "The neutralizer file has no neutralizer columns"
assert not neutralizers.duplicated(["numerai_ticker", "date"]).any()

validation = validation.merge(
    neutralizers, on=["numerai_ticker", "date"], how="left", validate="one_to_one"
)
del neutralizers
assert not validation[neutralizer_columns].isna().any().any(), "Missing validation neutralizers"

# Compute NCORR on each date
per_date_ncorr = validation.groupby("date").apply(
    lambda date_data: neutral_correlation(
        date_data[["prediction"]], date_data[neutralizer_columns], date_data["target"]
    )["prediction"], include_groups=False
)

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
per_date_ncorr.plot(title="Validation NCORR", kind="bar", ax=axes[0], xticks=[], legend=False, snap=False)
per_date_ncorr.cumsum().plot(title="Cumulative Validation NCORR", kind="line", ax=axes[1], legend=False)
plt.tight_layout()
plt.show()

Cumulative `NCORR` is easier to read than the raw per-date scores - think of it as a rough backtest of your model's neutralized target alignment over the sampled validation dates.

### NCORR summary metrics

A few summary metrics over the sampled validation dates:
- **Mean**: average score.
- **Sharpe**: consistency - mean divided by standard deviation.
- **Max drawdown**: the largest peak-to-trough drop in cumulative score.

In [ ]:
# Compute NCORR performance metrics
ncorr_mean = per_date_ncorr.mean()
ncorr_std = per_date_ncorr.std(ddof=0)
ncorr_sharpe = ncorr_mean / ncorr_std
ncorr_max_drawdown = (
    per_date_ncorr.cumsum().expanding(min_periods=1).max()
    - per_date_ncorr.cumsum()
).max()

pd.DataFrame({
    "mean": [ncorr_mean],
    "std": [ncorr_std],
    "sharpe": [ncorr_sharpe],
    "max_drawdown": [ncorr_max_drawdown]
}, index=["NCORR"]).T

### Benchmark contribution

We can also ask whether our model adds signal beyond a fixed public benchmark. We use `v6_lgbm_jupiter60` and neutralize both predictions and benchmark before comparing their contribution.

In [ ]:
# Download and join public benchmark predictions for the sampled dates
BENCHMARK_MODEL = "v6_lgbm_jupiter60"
napi.download_dataset(f"{DATA_VERSION}/validation_benchmark_models.parquet")
benchmarks = pd.read_parquet(
    f"{DATA_VERSION}/validation_benchmark_models.parquet",
    columns=["numerai_ticker", "date", BENCHMARK_MODEL],
    filters=[("date", "in", scoring_dates)],
)
validation = validation.merge(
    benchmarks, on=["numerai_ticker", "date"], how="left", validate="one_to_one"
)
del benchmarks

# Calculate contribution for dates covered by the benchmark
per_date_benchmark_contribution = (
    validation.dropna(subset=["prediction", "target", BENCHMARK_MODEL])
    .groupby("date")
    .apply(lambda date_data: neutral_contribution(
        date_data[["prediction"]],
        date_data[BENCHMARK_MODEL],
        date_data[neutralizer_columns],
        date_data["target"],
    )["prediction"], include_groups=False)
)

### Benchmark contribution validation performance

The fixed benchmark lets us estimate how much the model adds over the sampled validation dates:

In [ ]:
# Create a figure with two subplots side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
per_date_benchmark_contribution.plot(title="Validation benchmark contribution", kind="bar", ax=axes[0], xticks=[], legend=False, snap=False)
per_date_benchmark_contribution.cumsum().plot(title="Cumulative benchmark contribution", kind="line", ax=axes[1], legend=False)
plt.tight_layout()
plt.show()

### Benchmark contribution summary metrics

The table below summarizes contribution over dates covered by the benchmark.

In [ ]:
benchmark_summary = pd.DataFrame({
    "mean": [per_date_benchmark_contribution.mean()],
    "std": [per_date_benchmark_contribution.std(ddof=0)],
    "sharpe": [per_date_benchmark_contribution.mean() / per_date_benchmark_contribution.std(ddof=0)],
    "max_drawdown": [(per_date_benchmark_contribution.cumsum().cummax() - per_date_benchmark_contribution.cumsum()).max()],
}, index=["Benchmark contribution"]).T
benchmark_summary

These metrics don't need to be amazing - we're just getting started. The next tutorials cover improving target prediction while adding unique signal beyond public benchmarks.

## 3. Submissions

Unlike Kaggle competitions that evaluate models on <ins>test</ins> performance, Numerai evaluates models on <ins>live</ins> performance.

### Live predictions

New `live features` are released regularly - the current state of the stock market. Your job is to predict the target: stock-specific returns 60 business days into the future.

In [ ]:
# Download latest live features
napi.download_dataset(f"{DATA_VERSION}/live.parquet")

# Load live features
live_features = pd.read_parquet(f"{DATA_VERSION}/live.parquet", columns=feature_set)

# Generate live predictions
live_predictions = model.predict(live_features[feature_set])

# Format submission
pd.Series(live_predictions, index=live_features.index).rank(pct=True).to_frame("prediction")

### Model upload

To automate this process, you can simply:
- Define your prediction pipeline as a function
- Serialize your function using the `cloudpickle` library
- Upload your model pickle file to Numerai
- Let Numerai run your model to submit live predictions every day

Read more about Model Uploads and other self-hosted automation options in our [docs](https://docs.numer.ai/numerai-tournament/submissions#automation).

In [ ]:
# Define your prediction pipeline as a function
# Model Uploads expect a prediction column aligned to the input row index.
# TODO(verify): Test this contract with the final Classic v6 Model Upload runtime.
def predict(live_features: pd.DataFrame, live_benchmark_models: pd.DataFrame) -> pd.DataFrame:
    live_predictions = model.predict(live_features[feature_set])
    submission = pd.Series(live_predictions, index=live_features.index).rank(pct=True)
    return submission.to_frame("prediction")

In [ ]:
# Use the cloudpickle library to serialize your function
import cloudpickle
p = cloudpickle.dumps(predict)
with open("tutorial_basics.pkl", "wb") as f:
    f.write(p)

In [ ]:
# Download file if running in Google Colab
try:
    from google.colab import files
    files.download('tutorial_basics.pkl')
except ImportError:
    print('Model saved as tutorial_basics.pkl')

That's it! You now have a pickle file that is ready for upload.

Head back to the [Tournament Basics onboarding step](https://numer.ai/onboarding/tournament-basics) to upload your model!